In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [11]:
load_dotenv()

True

## Task 1: PromptTemplate

In [1]:
question = input('Please yype in your Question')

In [5]:
prompt = PromptTemplate(template='you are a Phd holder person Answer the following Question : What is your name and what do u do ?')

In [12]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [6]:
dynamic_prompt = PromptTemplate(template='you are a Phd holder person Answer the following Question {Question}',input_variables=['Question'])

In [13]:
finalprompt = dynamic_prompt.invoke({'Question': question})

In [14]:
Response = model.invoke(finalprompt)

In [16]:
Response.text

'As someone who has spent years studying the deep structures of formal systems, I absolutely love this question. It seems deceptively simple, but asking *why* $2 + 2 = 4$ (and not $2$, $6$, or $8$) actually takes us to the very bedrock of philosophy, logic, and pure mathematics. \n\nTo answer this from an academic perspective, we have to look at this question through three lenses: **linguistic semantics**, **formal axiomatic mathematics**, and **logical consistency**.\n\n---\n\n### 1. The Semantic Level: Symbols vs. Reality\nFirst, we must distinguish between the *symbols* we use and the *concepts* they represent. \n\nThe symbols "2", "4", "+", and "=" are arbitrary human inventions. If we, as a society, decided tomorrow to swap the names of the symbols "4" and "6", then $2 + 2$ would indeed equal $6$. \n\nHowever, the *quantity* would remain unchanged. If you have two apples ($\\bullet\\bullet$) and I give you two more apples ($\\bullet\\bullet$), you have a specific collection of app

## Task 2 : ChatPromptTemplate & Message Template

In [17]:
from langchain_core.prompts import ChatPromptTemplate

In [18]:
role = input()
query = input()

In [21]:
chat_temp = ChatPromptTemplate.from_messages([('system','''You are a Senior {role} with 1+ decade of experiance'''),('human','Help me understand this Query {query}')])

In [22]:
prompt = chat_temp.invoke({'role':role,'query':query})

In [23]:
chat_prompt_temp = model.invoke(prompt)

In [26]:
print(chat_prompt_temp.text)

As a teacher who has spent over a decade guiding students through the beautiful landscapes of mathematics, I absolutely love this question. 

Many people take $2 + 2 = 4$ for granted, assuming it is just a rule we must memorize. But asking *why* it is true—and why it couldn't be $6$, $8$, or any other number—takes us to the very foundation of human logic, language, and the nature of reality.

To understand why $2 + 2$ must equal $4$, we have to look at this question through three different lenses: **physical reality**, **linguistic agreement**, and **formal mathematical logic**.

---

### 1. The Physical Reality (The Empirical Argument)
At its most basic level, mathematics is a tool we invented to describe the universe. 

Imagine you are holding **two** apples in your left hand, and **two** apples in your right hand. If you place them all into a basket and count them one by one, you will always find exactly **four** apples. 

* **Why not 6 or 8?** Because physical objects cannot magica

# PART 2: Strectured Output using Pydantic

## Task 3: Pydantic Strecture output

In [41]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

In [28]:
class Answer(BaseModel):
    answer : str
    confidence : float
    source : str

In [29]:
parser = PydanticOutputParser(pydantic_object=Answer)

In [30]:
gemini = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [37]:
prompt = PromptTemplate(template='''You are a Senior {role}, answer the following Question {Question} in this {format}''',
                        input_variables=['role','Question'],partial_variables={'format':parser.get_format_instructions})

In [38]:
final_prompt = prompt.invoke({'role':'AI Engineer','Question':'which field is best Data Engineering or Gen AI Engineer'})

In [39]:
response = gemini.invoke(final_prompt)

In [45]:
gpt = ChatOpenAI(model='gpt-4o')

In [40]:
print(response.content)

[{'type': 'text', 'text': '```json\n{\n  "answer": "As a Senior AI Engineer, I evaluate both fields based on foundational longevity versus innovation velocity. Data Engineering is the indispensable backbone of the entire tech ecosystem, including AI. Without robust data pipelines, high-throughput storage, and clean data, Generative AI cannot function. It offers unparalleled job security, consistent market demand, and structured career progression. Generative AI Engineering, on the other hand, is at the absolute frontier of technology, focusing on LLMs, Retrieval-Augmented Generation (RAG), agentic workflows, and model fine-tuning. It currently commands higher salary premiums and offers the chance to build disruptive products, but it is highly volatile and susceptible to rapid tool obsolescence. Verdict: Choose Data Engineering if you prefer architectural stability, backend systems, and guaranteed long-term demand. Choose Gen AI Engineering if you thrive on rapid learning, uncertainty, 

In [46]:
result = gpt.invoke(final_prompt)

In [48]:
print(result.content)

```json
{
  "answer": "Both Data Engineering and Generative AI Engineering have their unique merits, and the choice depends on your interests and career goals. Data Engineering is critical for building robust data pipelines and infrastructure, which are foundational for any data-related field. It offers a stable career with a focus on optimizing data flow and storage. Generative AI Engineering is at the forefront of innovation, focusing on creating AI models that can generate content. It's a rapidly evolving field with potential for groundbreaking applications. If you enjoy working with data infrastructure, choose Data Engineering. If you're passionate about AI and its creative possibilities, Generative AI Engineering could be best.",
  "confidence": 0.85,
  "source": "Based on industry trends and job market analysis."
}
```


## Task 5: Simple Chain

In [50]:
chain = prompt | gpt | parser

In [51]:
response = chain.invoke({'role':'AI Engineer','Question':'Give me a joke on AI Engineer'})

In [59]:
print(response)

answer="Why do AI engineers prefer to work in the cloud? Because they find it hard to keep their feet on the ground when their heads are always in 'the cloud'!" confidence=0.9 source='Internet humor on AI'


## Task 6: Conditional Chain

In [60]:
age = input('Enter your Age')

In [107]:
from langchain_core.runnables import RunnableBranch,RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from typing import Literal
from pydantic import BaseModel,Field

In [108]:
class Sentiment(BaseModel):
    Fact : Literal['True','False'] = Field(description='return only True or False')

In [109]:
parser = PydanticOutputParser(pydantic_object=Sentiment)

In [110]:
conditional_prompt = PromptTemplate(template='You are a Fact Checker, check this {fact} and return the answer in this {format}',
                        input_variables=['fact'],partial_variables={'format':parser.get_format_instructions})

In [111]:
# fact_prompt = conditional_prompt.invoke({'fact':'Modi is the PM of india'})

In [112]:
chain_gemini = ChatGoogleGenerativeAI(model='gemini-3.5-flash')
chain_gpt = ChatOpenAI(model='gpt-4o')

In [113]:
# res = chain_gemini.invoke(fact_prompt)

In [114]:
# res.content

In [115]:
chain_1 = conditional_prompt | chain_gemini | parser

In [116]:
true_prompt = PromptTemplate(template='Generate the Reason why its {fact}',input_variables=['fact'])
false_prompt = PromptTemplate(template='Generate the Reason why its {fact}',input_variables=['fact'])

In [117]:
strParser = StrOutputParser()

In [102]:
chain_2 = true_prompt | chain_gpt | strParser
chain_3 = true_prompt | chain_gpt | strParser

In [118]:
conditional_chain = RunnableBranch((lambda x:x.Fact=='True',chain_2),(lambda x : x.Fact=='False',chain_3),
                                   RunnableLambda(lambda x : "Sorry cant confirm your Fact is True or False"))

In [119]:
final_chain= chain_1 | conditional_chain

In [120]:
result = final_chain.invoke({'fact':"Modi is the PM of India"})

In [121]:
result

'To provide a meaningful analysis about why a statement is factually true, detailed context and specifics about the statement in question are necessary. Without a specific statement to evaluate, I can only give you a general approach to understanding why something might be considered true:\n\n1. **Empirical Evidence**: The statement is supported by empirical data or evidence collected through observation, experimentation, or reliable reporting.\n\n2. **Logical Consistency**: The statement is logically consistent with established knowledge and does not contradict proven theories or facts.\n\n3. **Expert Consensus**: There is a consensus among experts in the relevant field supporting the statement as true based on their studies and professional assessment.\n\n4. **Reproducibility**: The claim or statement can be tested and repeatedly verified by different independent sources or researchers, yielding the same results.\n\n5. **Source Reliability**: The statement originates from a credible 

In [126]:
print(final_chain.get_graph().draw_ascii())

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      


## Task 7: Parallel Chain

In [217]:
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from langchain_core.runnables import RunnableParallel

In [218]:
par_model_1 = ChatOpenAI(model='gpt-3.5-turbo')
par_model_2 = ChatOpenAI(model='gpt-4o')
par_model_3 = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')


In [219]:
strrParser = StrOutputParser()

In [220]:
ans_prompt = PromptTemplate(template='Generate the Answer on the {topic}',input_variables=['topic'])
sum_prompt = PromptTemplate(template='Generate the Summary on the {topic} in bullet points',input_variables=['topic'])
quiz_prompt = PromptTemplate(template='Generate the follow up Questions on the {topic} for practice',input_variables=['topic'])

In [221]:
par_chain_1 = ans_prompt | par_model_1 | strrParser
par_chain_2 = sum_prompt | par_model_2 | strrParser
par_chain_3 = quiz_prompt | par_model_3 | strrParser

In [222]:
parallel_chain = RunnableParallel({'answer':ans_prompt | par_model_1 | strrParser,'summary':par_chain_2,'quiz':par_chain_3})

In [223]:
final_model = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [224]:
class QuestionAnswer(BaseModel):
    Question : str =Field(description='Return only the exact Question which is asked ')
    answer : str = Field(description='return Only the Answer')
    summary : str = Field(description='Only Return the summary from the Question asked')
    quiz : list[str] = Field(description='Only Return the 5 follow-up Question You are asking')

pyParser = PydanticOutputParser(pydantic_object=QuestionAnswer)

In [225]:
final_prompt = PromptTemplate(template='Marge the following {answer}, {summary} and {quiz} together and return the result in this {format}',input_variables=['answer','summary','quiz'],partial_variables={'format':pyParser.get_format_instructions})

In [226]:
single_chain = final_prompt | final_model | pyParser

In [227]:
final_chain = parallel_chain | single_chain

In [228]:
parallel_chain_resul= final_chain.invoke({'topic':"10 + 10 = ?"})

In [229]:
print(parallel_chain_resul)

Question='10 + 10 = ?' answer='20' summary="The equation 10 + 10 = 20 is a basic arithmetic problem involving addition, where the '+' symbol signifies combining the two numbers to find their total. Understanding simple addition is fundamental for daily tasks like budgeting and measurement, serves as a stepping stone to advanced mathematics, and is key to developing mental math fluency." quiz=['If 10 + 10 = 20, what is 10 + 11?', 'What is 20 + 20?', 'If you have 20 and you add 10, what is the new total?', '10 + ___ = 20', 'You have 10 red apples and 10 green apples. How many apples do you have in total?']


In [230]:
for i in parallel_chain_resul.quiz:
    print(i,end='\n')

If 10 + 10 = 20, what is 10 + 11?
What is 20 + 20?
If you have 20 and you add 10, what is the new total?
10 + ___ = 20
You have 10 red apples and 10 green apples. How many apples do you have in total?


In [210]:
# print(final_chain.get_graph().draw_ascii())

In [ ]:
# import streamlit as st 

# st.title(" Parallel Chain Model")

# question = st.text_input('Type in your Question ?')



# PART 4 - Runnables & LCEL

## Task 8:  Rubnnables Basics

## Task 9: LCEL - Based RAG Chain

## Task 10: Observations & Insights

### Answer 1 : 
Structured Output are important so we can get the outputs in structured format from the LLMs and save then in a structured format in a Database for easy retrival, search, etc. 

### Answer 2 : 

Advantages of LCEL:
    \n 1. No need to import runnable, runnable sequence to define the Chain flow
    \n 2. just use pipe operator ('|') to run define the chain flow
    \n eg : Chain = prompt | llm | Parser
    \n 3. low code """

### Answer 3 :
Parallel Chain : Use it when you want to perform multiple tasks Simintunatioly
Conditional Chain : Used when a certain chain should be execute Based on conditions or answer we receive. 